In [1]:
import pickle

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn import metrics
from sklearn.model_selection import train_test_split

In [3]:
FILE_PATH = "wildfire_data.csv"

df = pd.read_csv(FILE_PATH)

In [5]:
df.head()

,temperature,NDVI,humidity,wind_speed,slope,fire_risk
0,31.236204,0.311080,35.702341,17.472169,5.719959,0
1,48.521429,0.525141,34.818728,20.323672,8.054323,0
2,41.959818,0.723768,74.375275,7.760762,7.601609,0
3,37.959755,0.639335,34.972772,16.372104,1.538999,0
4,24.680559,0.683937,36.316984,15.150158,1.492495,0


In [7]:
df.shape

(1000, 6)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   temperature  1000 non-null   float64
 1   NDVI         1000 non-null   float64
 2   humidity     1000 non-null   float64
 3   wind_speed   1000 non-null   float64
 4   slope        1000 non-null   float64
 5   fire_risk    1000 non-null   int64  
dtypes: float64(5), int64(1)
memory usage: 47.0 KB


In [13]:
feature_cols = ["temperature", "NDVI", "humidity", "wind_speed", "slope"]

X = df[feature_cols]
y = df["fire_risk"]

In [15]:
X.head()

,temperature,NDVI,humidity,wind_speed,slope
0,31.236204,0.311080,35.702341,17.472169,5.719959
1,48.521429,0.525141,34.818728,20.323672,8.054323
2,41.959818,0.723768,74.375275,7.760762,7.601609
3,37.959755,0.639335,34.972772,16.372104,1.538999
4,24.680559,0.683937,36.316984,15.150158,1.492495


In [17]:
X.shape

(1000, 5)

In [23]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: fire_risk, dtype: int64

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [27]:
#class imbalance ratio for fire_risk
n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())

imbalance_ratio = max(n_neg, n_pos) / min(n_neg, n_pos) if min(n_neg,n_pos) > 0 else np.inf

In [31]:
#positive weight required for XGBoost 
scale_pos_weight = n_neg / n_pos if n_pos > 0 else np.nan

In [33]:
n_neg

772

In [35]:
n_pos

28

In [37]:
imbalance_ratio

27.571428571428573

In [39]:
scale_pos_weight

27.571428571428573

In [41]:
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss",
)
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [43]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

In [45]:
acc = metrics.accuracy_score(y_test, y_pred)
roc_auc = metrics.roc_auc_score(y_test, y_proba)
cm = metrics.confusion_matrix(y_test, y_pred)

In [47]:
print("Test set evaluation")
print("-" * 40)
print(f"Accuracy   : {acc:.4f}")
print(f"ROC-AUC    : {roc_auc:.4f}")
print()
print("Confusion matrix (rows = true, cols = predicted):")
print(cm)

Test set evaluation
----------------------------------------
Accuracy   : 1.0000
ROC-AUC    : 1.0000

Confusion matrix (rows = true, cols = predicted):
[[193   0]
 [  0   7]]


In [49]:
MODEL_PATH = 'model.pkl'
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(model,f)

In [51]:
FEATURE_COLS = ["temperature", "NDVI", "humidity", "wind_speed", "slope"]

with open("model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

def predict_risk(input_dict):
    """Predict fire risk from a single observation as a dict of feature names → values."""
    X_new = pd.DataFrame([input_dict])[FEATURE_COLS]
    proba = float(loaded_model.predict_proba(X_new)[0, 1])

    if proba < 0.3:
        risk = "Low"
    elif proba <= 0.7:
        risk = "Medium"
    else:
        risk = "High"

    return proba, risk

In [53]:
sample = {
    "temperature": 32.5,
    "NDVI": 0.35,
    "humidity": 28.0,
    "wind_speed": 18.5,
    "slope": 12.0,
}

prob, risk = predict_risk(sample)

print("Sample prediction")
print("-" * 40)
print(f"Inputs: {sample}")
print(f"P(fire risk): {prob:.3f}")
print(f"Category:     {risk}")

Sample prediction
----------------------------------------
Inputs: {'temperature': 32.5, 'NDVI': 0.35, 'humidity': 28.0, 'wind_speed': 18.5, 'slope': 12.0}
P(fire risk): 0.012
Category:     Low
